In [1]:
from pathlib import Path

# Encontramos los paths para los datos
root = Path('..').resolve() # ruta absoluta

data = root / "data"
data_raw = data / "raw"

# ref_path = data_raw / "data_ref_until_2020-02-13.csv"
gas_path = data_raw / "database_gas.csv"
pos_path = data_raw / "database_pos.csv"

In [2]:
import pandas as pd

tipos_gas = {
    'CO2CosIRValue': 'uint16',
    'CO2MG811Value': 'uint16',
    'MOX1': 'uint16',
    'MOX2': 'uint16',
    'MOX3': 'uint16',
    'MOX4': 'uint16',
    'COValue': 'uint16'
}

# Creamos los dataframes
gas = pd.read_csv(gas_path, dtype=tipos_gas, parse_dates=['timestamp'])
pos = pd.read_csv(pos_path, parse_dates=['datetime'])
pos = pos.fillna(0.0)

In [3]:
# Los datos de ref se corresponde con los últimos del gas

In [4]:
gas.head()

,timestamp,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue
0,2019-11-06 11:37:13.038174+01:00,19.48,54.86,128,563,476,731,649,565,128
1,2019-11-06 11:37:32.744996+01:00,19.59,54.23,129,563,477,731,649,565,125
2,2019-11-06 11:37:53.018087+01:00,19.63,54.05,128,566,478,732,649,565,125
3,2019-11-06 11:38:13.093151+01:00,19.64,53.74,128,566,478,732,649,565,125
4,2019-11-06 11:38:33.032832+01:00,19.67,53.53,128,569,480,732,650,565,125


In [5]:
pos.head()

,datetime,Living room,Bedroom,Bathroom,Kitchen,Hallway
0,2019-11-01 02:52:55.271086300+00:00,0.0,0.0,0.0,0.0,0.0
1,2019-11-01 03:19:57.417067700+00:00,0.0,1.0,0.0,0.0,0.0
2,2019-11-01 03:21:53.257070700+00:00,0.0,0.0,0.0,0.0,0.0
3,2019-11-01 05:11:08.874031+00:00,0.0,1.0,0.0,0.0,0.0
4,2019-11-01 05:12:38.437033500+00:00,0.0,0.0,0.0,0.0,0.0


Notamos dos cosas:
- Las fechas no están apuntadas bajo la misma zona horaria: Las del dataframe gas están en UTC+1 mientras que las de posición están en UTC. Debemos uniformizar y convertirlas todas a UTC por ejemplo.

- La instalación de los sensores de gas se produjo 5 días despues que la de los sensores de posición. Como queremos predecir posición en función de gas vamos a filtrar los datos de posición para que se nos muestren solo a partir de la fecha de instalación de los sensores de gas.

In [6]:
# Convertimos a UTC
gas['timestamp'] = gas['timestamp'].dt.tz_convert('UTC')

In [7]:
# Filtramos los datos de pos
instalacion_gas = gas.iloc[0,0]

pos = pos[pos['datetime'] >= instalacion_gas]

Aún así las horas no coinciden:

In [8]:
gas.head(1)

,timestamp,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue
0,2019-11-06 10:37:13.038174+00:00,19.48,54.86,128,563,476,731,649,565,128


In [9]:
pos.head(1)

,datetime,Living room,Bedroom,Bathroom,Kitchen,Hallway
1518,2019-11-06 10:41:36.160021+00:00,0.0,0.0,1.0,0.0,0.0


Hay una diferencia de unos tres minutos, es más no coinciden exactamente. Podemos ver cuántas mediciones de gas se hicieron a las 10:41 del día 6 de noviembre de 2019 y ver que ninguna hora coincide:

In [10]:
gas.set_index('timestamp').sort_index().loc['2019-11-06- 10:41']

,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue
timestamp,,,,,,,,,
2019-11-06 10:41:13.029570+00:00,19.66,53.51,127,574,488,732,650,566,118
2019-11-06 10:41:32.886953+00:00,19.66,53.51,127,574,489,732,650,566,118
2019-11-06 10:41:53.056103+00:00,19.66,53.77,128,574,490,732,650,566,118


Las dos mediciones más cercanas difieren en 4 segundos. Lo importante es que vemos que las mediciones son muy parecidas (tiene sentido pues se tomaron muy cerca). Para no complicar mucho la integración de ambos dataframes lo que haremos será redondear las mediciones al minuto y quedarnos con la primera.

Queremos intentar localizar en que habitación se produjo actividad en función de los gases. El problema es que hay mediciones de posición en las que no está registrada la habitación. Esto ocurre por el funcionamiento del sensor. Este se registra __movimiento__, no puede saber dónde está. Por ejemplo, si algo se movió en la cocina, lo detectará y registrará un 1. Pero si después esa persona no sigue moviéndose no sigue registrando 1 sino que registrará 0 porque ya no detecta más movimiento. Vamos a quedarnos con las observaciones en las que alguna de las habitaciones tenga un 1:

In [11]:
habitaciones = ['Living room', 'Bedroom', 'Bathroom', 'Kitchen', 'Hallway']
activity_mask = pos[habitaciones].sum(axis=1).astype(bool)
pos = pos[activity_mask]

In [12]:
# Creamos una variable que registre la habitación donde se produjo la actividad
pos['room'] = pos[habitaciones].idxmax(axis=1)
pos['room'] = pos['room'].astype('category')

In [13]:
# Resolución de 1 segundo
pos['timestamp_rounded'] = pos['datetime' ].dt.floor('30s')
gas['timestamp_rounded'] = gas['timestamp'].dt.floor('30s')
# Eliminamos duplicados
pos_no_dup = pos.drop_duplicates(subset=['timestamp_rounded'], keep='first')
gas_no_dup = gas.drop_duplicates(subset=['timestamp_rounded'], keep='first')

In [14]:
pos_no_dup.shape

(9526, 8)

In [15]:
gas_no_dup.shape

(277434, 11)

In [16]:
final_df = pd.merge(
    pos_no_dup.drop(columns=['datetime']+habitaciones),
    gas_no_dup.drop(columns=['timestamp']),
    on='timestamp_rounded',
    how='inner',
    validate='1:1'
)

final_df = final_df.set_index('timestamp_rounded').sort_index()

In [17]:
final_df

,room,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue
timestamp_rounded,,,,,,,,,,
2019-11-06 10:41:30+00:00,Bathroom,19.66,53.51,127,574,489,732,650,566,118
2019-11-06 10:43:00+00:00,Hallway,19.69,53.77,128,578,496,732,650,566,116
2019-11-06 10:43:30+00:00,Living room,19.66,53.82,128,578,496,732,650,566,116
2019-11-06 10:44:30+00:00,Living room,19.60,53.87,128,579,496,732,650,565,114
2019-11-06 10:45:00+00:00,Hallway,19.48,54.25,128,581,497,732,649,565,114
...,...,...,...,...,...,...,...,...,...,...
2020-02-12 14:13:00+00:00,Hallway,20.64,58.38,44,518,520,692,633,579,173
2020-02-12 14:14:30+00:00,Living room,20.69,58.25,44,517,520,693,635,580,173
2020-02-12 14:15:30+00:00,Living room,20.69,57.75,45,517,522,695,637,584,172


In [18]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 9485 entries, 2019-11-06 10:41:30+00:00 to 2020-02-12 14:19:00+00:00
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   room           9485 non-null   category
 1   temperature    9485 non-null   float64 
 2   humidity       9485 non-null   float64 
 3   CO2CosIRValue  9485 non-null   uint16  
 4   CO2MG811Value  9485 non-null   uint16  
 5   MOX1           9485 non-null   uint16  
 6   MOX2           9485 non-null   uint16  
 7   MOX3           9485 non-null   uint16  
 8   MOX4           9485 non-null   uint16  
 9   COValue        9485 non-null   uint16  
dtypes: category(1), float64(2), uint16(7)
memory usage: 361.5 KB


Creamos una carpeta para los datos procesados si no existe y sino los sobreescribimos:

In [19]:
processed_folder = data / "processed_data"

processed_folder.mkdir(parents=True, exist_ok=True) # Crea si no existe y sino no pasa nada

final_df.to_csv(processed_folder / "multilabel_data.csv")    